# Этап C: обучающая выборка, dev, синтетический тест, судья

Перед запуском:

* **+ Add Input** → датасет `ru-fin-retrieval` (код и корпус) **и** датасет
  с набором запросов (`queries.jsonl`, `split.json`, `pool_candidates.tsv`);
* **Settings** → **Accelerator** → **GPU T4 ×2**, интернет включён;
* **токен HuggingFace подключён к этой тетради.** Секреты Kaggle
  привязываются к конкретной тетради через браузер и новым ядром
  не наследуются. Один раз проект на этом потерял два часа квоты:
  без токена скачивание весов режется по скорости и может встать намертво.

Порядок действий целиком — `kaggle/README.md`.

## 1. Карта и токен

Должна быть видна Tesla T4. Токен проверяется до всякого долгого прогона.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

from kaggle_secrets import UserSecretsClient
try:
    token = UserSecretsClient().get_secret("HF_TOKEN")
    import os
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = token
    print("токен HuggingFace на месте, длина", len(token))
except Exception as e:
    print("ТОКЕНА НЕТ:", e)
    print("Add-ons -> Secrets -> HF_TOKEN. Без него скачивание весов "
          "режется по скорости.")

## 2. Библиотеки

vLLM вместо `model.generate`: на пятнадцати тысячах вопросов разница
в скорости — порядок. `sentence-transformers` нужен третьему фильтру
(кросс-энкодер-учитель).

Установка vLLM тянет свою версию torch и занимает несколько минут.

In [ ]:
!pip install -q -U vllm
!pip install -q -U sentence-transformers

## 3. Код и данные на место

In [ ]:
import glob, os, shutil, zipfile

# Kaggle монтирует датасеты по-разному: то прямо в /kaggle/input/<имя>,
# то глубже, внутри /kaggle/input/datasets/<владелец>/<имя>. Поэтому путь
# не задаётся, а источник ищется по содержимому: нужен каталог, где лежит
# пакет rufin или корпус. Глубина ограничена, чтобы не обходить всё подряд.
def find_source(root='/kaggle/input', max_depth=4):
    found, archives = [], []
    for cur, dirs, files in os.walk(root):
        if cur.count('/') - root.count('/') >= max_depth:
            dirs.clear()
        if 'rufin' in dirs:
            stale = ('__notebook__.ipynb' in files or 'embeddings' in dirs
                     or '__output__.json' in files)
            found.append((1 if stale else 0, cur))
        archives += [os.path.join(cur, f) for f in files if f.endswith('.zip')]
    if found:
        found.sort()
        return found[0][1], None
    return (os.path.dirname(archives[0]), archives[0]) if archives else (None, None)


def show_tree(root='/kaggle/input', max_depth=3):
    for cur, dirs, files in os.walk(root):
        depth = cur.count('/') - root.count('/')
        if depth >= max_depth:
            dirs.clear()
        print('   ' * depth + os.path.basename(cur) + '/')
        for f in sorted(files)[:6]:
            print('   ' * (depth + 1) + f)


src, archive = find_source()
if src is None:
    print('что ноутбук видит на входе:')
    show_tree()
    raise SystemExit('датасет не подключён: панель справа -> + Add Input -> ru-fin-retrieval -> +')

print('источник:', src)
if archive:
    print('лежит архивом, распаковываю:', archive)
    with zipfile.ZipFile(archive) as z:
        z.extractall('/kaggle/working')
else:
    for item in glob.glob(os.path.join(src, '*')):
        dst = os.path.join('/kaggle/working', os.path.basename(item))
        if os.path.isdir(item):
            shutil.copytree(item, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(item, dst)

%cd /kaggle/working
print('\nв рабочей папке:', sorted(os.listdir('.')))

assert os.path.isdir('rufin'), f'пакета rufin нет; в источнике было: {sorted(os.listdir(src))}'
# Kaggle иногда разжимает архив при создании датасета, имя корпуса плавает
assert glob.glob('acts.jsonl*'), 'корпуса нет'
print('всё на месте:', sorted(glob.glob('acts.jsonl*')))


In [ ]:
import glob
# сплит обязателен: без него генерация уйдёт на акты теста
print('сплит:', glob.glob('/kaggle/input/**/split.json', recursive=True)
      or glob.glob('split.json'))
print('запросы:', glob.glob('/kaggle/input/**/queries.jsonl', recursive=True)
      or glob.glob('queries.jsonl'))
print('пул:', glob.glob('/kaggle/input/**/pool_candidates.tsv', recursive=True)
      or glob.glob('pool_candidates.tsv'))

## 4. Замер скорости

Сотня фрагментов, три стиля. Считает, сколько вопросов в секунду даёт
модель и во что это выльется на десяти и пятнадцати тысячах. **Объём
обучающей выборки назначается по этому числу, а не по плану:** десять
тысяч за три часа лучше пятнадцати, которые не досчитаются к концу
сессии.

Двадцать минут, включая скачивание весов.

In [ ]:
!cd /kaggle/working && python run_phase_c.py --probe 100

## 5. Пробный прогон

Сто актов, по двадцать вопросов на выборку. Проверяем, что генерация,
фильтры и запись работают, и смотрим на сами вопросы глазами. Десять
минут — дешевле, чем узнать о браке через три часа.

In [ ]:
!cd /kaggle/working && python run_phase_c.py \
    --limit-acts 100 --probe 0 --train 20 --dev 20 --test 20 \
    --out /kaggle/working/proba

import json
for имя in ('synthetic_train', 'synthetic_dev', 'synthetic_test'):
    path = f'/kaggle/working/proba/queries/{имя}.jsonl'
    try:
        rows = [json.loads(l) for l in open(path, encoding='utf-8')]
    except FileNotFoundError:
        print(имя, '— файла нет'); continue
    print(f'\n=== {имя}: {len(rows)} ===')
    for r in rows[:6]:
        print(f"  [{r['style']}] {r['text'][:110]}")

## 6. Полный прогон

Объём в `--train` подставляется **по замеру из шага 4**. Третий фильтр
(`--teacher-filter`) считается отдельно и стоит дорого: `вопросов ×
глубина` проходов кросс-энкодера. Если по замеру видно, что в сессию
не укладывается, сначала прогоняются генерации, а фильтр — вторым
запуском: шаги независимы и готовое не пересчитывают.

Лучше не держать вкладку открытой: **Save Version** → **Save & Run All
(Commit)**, Kaggle выполнит всё у себя.

In [ ]:
!cd /kaggle/working && python run_phase_c.py --probe 0 \
    --train 10000 --dev 300 --test 200 --teacher-filter

## 7. Разметка пула судьёй

Отдельным запуском: судье нужна модель крупнее генератора, и держать обе
в памяти незачем. Нужен `pool_candidates.tsv` — он собирается локально
(`make pool`) **после** прогона всех конфигураций этапа B по нынешнему
набору запросов.

In [ ]:
!cd /kaggle/working && python run_phase_c.py --probe 0 --judge

## 8. Что получилось

In [ ]:
import json
r = json.load(open('/kaggle/working/report_phase_c.json', encoding='utf-8'))
for шаг, v in r.items():
    print(f'--- {шаг}')
    print('   ', v if not isinstance(v, dict) else
          {k: x for k, x in v.items() if not isinstance(x, dict)})

## 9. Сохранить и скачать

**Save Version** → **Save & Run All (Commit)**.

Скачать нужно только маленькое:

* `queries/synthetic_train.jsonl`, `synthetic_dev.jsonl`,
  `synthetic_test.jsonl` — положить в `data/queries/`;
* `queries/judge.tsv` — туда же;
* `report_phase_c.json` — в `docs/raw/`.

Дальше локально: `make queries`, `make judge`, `make sample-for-human`,
`make kappa`, `make bench`.